# VisDrone — yolo26**s**

Lam tron mot size: baseline -> prune 50% -> finetune + CWD -> val.
Ket qua la **hai dong** cua bang: `YOLO26-S` va `Ours-S`.

| | |
|---|---|
| Baseline | `yolo26s.pt` (COCO), 100 epoch tren VisDrone |
| Ours | L1-norm uniform 50% (div 8) + 100 epoch CWD tau=9, kd_layers=neck |
| Batch / imgsz / seed | 16 / 640 / 0 |
| cos_lr / patience / warmup | False / 100 / 3.0 |
| Uoc tinh | ~6h, **1 phien** |

> Bon notebook n/s/m/l dung **y het** cac tham so nay. Doi mot cai thoi la
> ca bang het so sanh duoc. Dung sua `EPOCHS`, `BATCH`, `IMGSZ`, `RATIO`.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
   (can Internet: VisDrone 2.3 GB va `yolo26s.pt` deu tai tu mang)
2. Bam **Save & Run All**. Lan dau khong can Add Data.
3. Phien tu dung o 10h. Cell cuoi bao con thieu bao nhieu epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai. Lap den khi bao `XONG`.
4. Xong thi gui lai bang 2 dong o cell cuoi.

Notebook chay tuan tu baseline roi moi den Ours. Neu het gio giua chung, cac
cell sau tu bo qua va in ra con thieu gi — khong bao loi.

### Neu phien bi danh dau **failed**

Kaggle **khong luu output** cua version bi giet vi qua gio, nen "Add Data ->
Your Work" se khong thay no. Van lay lai duoc:

1. Mo version bi failed -> tab **Output** -> tai `last.pt` ve.
   Trinh duyet doi duoi thanh `.zip` (file `.pt` cua PyTorch von la mot zip).
   **Doi ten lai thanh `.pt`, KHONG giai nen.**
2. Upload thanh Kaggle dataset.
3. **Add Data** dataset do, roi dien duong dan file vao `MANUAL_LAST` o cell 2:

```python
MANUAL_LAST = {
    BASE_NAME: "/kaggle/input/vd-resume/last.pt",
    OURS_NAME: "",
}
```

Neu ban upload ca thu muc (giu nguyen ten `vd_yolo26m/weights/last.pt`) thi
khong can dien gi — cell resume tu tim thay.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [1]:
import os, sys, glob, shutil, pathlib, subprocess

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "pruning"))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
from ultralytics import YOLO
print("GPU:", torch.cuda.device_count())

Cloning into '/kaggle/working/yolo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.6.0 which is incompatible.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: 2


## 2. Cau hinh

In [2]:
SIZE   = "s"
MODEL  = "yolo26s.pt"
RATIO  = 0.5

DATA   = "VisDrone.yaml"   # Ultralytics tu tai 2.3 GB, can Internet: On
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640
DEVICE = "0,1" if torch.cuda.device_count() > 1 else "0"
STOP_AFTER_H = 10.0        # tu dung truoc moc 12h de output kip luu

# Chot cung ba tham so nay thay vi de mac dinh, vi cac run VOC truoc day KHONG
# dong nhat: n/s/l chay batch 32 + cos_lr=True + patience 20-30, con m chay
# batch 16 + cos_lr=False + patience 100. Bang theo size ma moi size mot config
# thi khong so sanh duoc. Lay config cua m vi do la cau hinh chinh cua bai.
# patience=100 = tat early stop -> ca 4 size deu la run 100 epoch that.
COS_LR   = False
PATIENCE = 100
WARMUP   = 3.0

BASE_NAME = "vd_yolo26s"
OURS_NAME = "vd_ourss"
PRUNED = REPO_DIR / "weights" / "yolo26s_vd_pruned50.pt"

# Chi dung khi phien bi Kaggle danh dau FAILED: version do khong tu luu output
# nen "Add Data -> Your Work" se khong co no. Vao tab Output cua version do,
# tai last.pt ve (trinh duyet doi duoi thanh .zip - **doi ten lai thanh .pt**,
# KHONG giai nen), upload thanh dataset roi dan duong dan file vao day.
MANUAL_LAST = {
    BASE_NAME: "",   # vd "/kaggle/input/vd-resume/last.pt"
    OURS_NAME: "",
}

print(BASE_NAME, "|", OURS_NAME, "|", DEVICE)

vd_yolo26s | vd_ourss | 0,1


## 3. Ham dung chung

In [3]:
# Hai ham nho dung chung cho ca hai lan train duoi day.

def epochs_of(run_dir):
    """So epoch da train xong cua mot thu muc run.

    Doc results.csv truoc; neu khong co (vd chi tai moi last.pt ve tu mot phien
    bi Kaggle danh dau failed) thi doc thang so epoch trong checkpoint.
    """
    run_dir = pathlib.Path(run_dir)
    csv = run_dir / "results.csv"
    if csv.exists():
        rows = [r for r in csv.read_text().strip().splitlines()[1:] if r.strip()]
        if rows:
            return int(float(rows[-1].split(",")[0]))
    last = run_dir / "weights" / "last.pt"
    if last.exists():
        try:
            ck = torch.load(last, map_location="cpu", weights_only=False)
            ep = int(ck.get("epoch", -1))
            total = int((ck.get("train_args") or {}).get("epochs", 0) or 0)
            del ck
            return (ep + 1) if ep >= 0 else total
        except Exception:
            pass
    return 0


def done_epochs(name):
    return epochs_of(REPO_DIR / "runs" / name)


def train(name, weights, **extra):
    """Train moi, hoac train TIEP neu phien truoc bi cat ngang.

    Co last.pt la resume. Khong lay so epoch lam dieu kien: doc that bai thi se
    am tham train lai tu dau, mat ca chuc gio ma khong bao gi.
    """
    last = REPO_DIR / "runs" / name / "weights" / "last.pt"
    if last.exists():
        print("[{}] resume tu epoch {}".format(name, done_epochs(name)))
        YOLO(str(last)).train(resume=True, stop_after_h=STOP_AFTER_H)
    else:
        print("[{}] train tu dau".format(name))
        YOLO(weights).train(data=DATA, epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                            device=DEVICE, seed=0,
                            cos_lr=COS_LR, patience=PATIENCE,
                            warmup_epochs=WARMUP,
                            project=str(REPO_DIR / "runs"), name=name,
                            exist_ok=True, stop_after_h=STOP_AFTER_H, **extra)
    return done_epochs(name)

## 4. Resume

In [4]:
# Kaggle giet phien o 12h, va phien BI GIET thi khong luu output -> mat sach
# last.pt. Vi the moi phien tu dung o STOP_AFTER_H roi ket thuc binh thuong.
# Lan sau: Add Data -> Your Work -> output lan truoc, cell nay chep runs/ ve.
def bring_back(name):
    """Tim checkpoint cu trong /kaggle/input va chep ve runs/<name>."""
    dst = REPO_DIR / "runs" / name
    if dst.exists():
        return

    cands = []
    # (a) Add Data output cua lan chay truoc: .../yolo/runs/<name>/
    cands += glob.glob("/kaggle/input/**/runs/" + name, recursive=True)
    # (b) Upload thu cong: thu muc ten <name> nam bat ky dau, khong can co runs/.
    cands += glob.glob("/kaggle/input/**/" + name, recursive=True)
    cands = [c for c in set(cands)
             if pathlib.Path(c, "weights", "last.pt").exists()]

    if cands:
        # Gan nhieu output (phien 1, phien 2, ...) thi phai lay ban NHIEU EPOCH
        # NHAT. Lay "cai dau tien tim thay" la sai: thu tu glob khong xac dinh,
        # co the chep nham ban cu va mat vai gio train ma khong he biet.
        best = max(cands, key=epochs_of)
        shutil.copytree(best, dst)
        print("  {}: {} epoch  <- {}".format(name, epochs_of(best), best))
        return

    # (c) Chi co moi file last.pt roi le -> dan duong dan vao MANUAL_LAST.
    man = MANUAL_LAST.get(name, "")
    if man and pathlib.Path(man).exists():
        (dst / "weights").mkdir(parents=True, exist_ok=True)
        shutil.copy2(man, dst / "weights" / "last.pt")
        print("  {}: {} epoch  <- (thu cong) {}".format(name, epochs_of(dst), man))
    elif man:
        print("  {}: !! MANUAL_LAST tro toi file khong ton tai: {}".format(name, man))


for name in (BASE_NAME, OURS_NAME):
    bring_back(name)

# Model da prune cung chep ve, neu khong se prune lai (khong sai, chi ton them).
if not PRUNED.exists():
    for src in glob.glob("/kaggle/input/**/" + PRUNED.name, recursive=True):
        PRUNED.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PRUNED)
        print("  {} <- {}".format(PRUNED.name, src))
        break

print()
print("baseline:", done_epochs(BASE_NAME), "/", EPOCHS, "epoch")
print("ours    :", done_epochs(OURS_NAME), "/", EPOCHS, "epoch")


baseline: 0 / 100 epoch
ours    : 0 / 100 epoch


## 5. Baseline yolo26s

In [5]:
n_base = train(BASE_NAME, MODEL)

if n_base < EPOCHS:
    print()
    print("Baseline moi {}/{} epoch - het gio phien nay.".format(n_base, EPOCHS))
    print("Add Data output lan nay roi Save & Run All lai. Cac cell duoi se bo qua.")

[vd_yolo26s] train tu dau
New https://pypi.org/project/ultralytics/8.4.163 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
                                                      CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, cwd_learnable_tau_init=9.0, cwd_learnable_tau_lr=0.001, cwd_projection=False, cwd_tau_reg=0.1, cwd_temperature=9.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dms=False, dms_decay_ratio=1, dms_grad_scale=-1.0, dms_lambda=1.0, dms_lr=2e-05, dms_target=0.3, dms_taylor_type=taylor, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, finetune=Fals

## 6. Prune 50%

In [6]:
# Goi thang cac ham trong pruning/, khong qua script.
from prune_common import load_and_prepare, create_masks, finalize_pruning
from prune_l1norm import compute_l1norm_importance

BEST_BASE = REPO_DIR / "runs" / BASE_NAME / "weights" / "best.pt"

if n_base < EPOCHS:
    print("Bo qua: baseline chua xong.")
elif PRUNED.exists():
    print("Da co", PRUNED.name)
else:
    PRUNED.parent.mkdir(parents=True, exist_ok=True)
    m0, bn_dict, ignore_bn, _chunk, layer_cfg, pruned_yaml = load_and_prepare(
        str(BEST_BASE), str(REPO_DIR / "cfg" / "yolo26m.yaml"), SIZE, None)
    imp = compute_l1norm_importance(m0, bn_dict, ignore_bn)
    masks = create_masks(imp, m0, ignore_bn, layer_cfg, RATIO, 8)
    out = finalize_pruning(m0, masks, pruned_yaml, ignore_bn, str(BEST_BASE),
                           str(REPO_DIR / "weights"), 8, RATIO,
                           method_name="l1norm")
    pathlib.Path(out).replace(PRUNED)
    print("->", PRUNED)

Step 1: Thu thập BatchNorm layers...
  Tổng BN layers: 114
  Ignore (residual): 32
  Chunk constraint: 0
  Prunable BN layers: 82

Step 6: Tạo pruned model config...
  nc: 10, scale: s
  Backbone layers: 11
  Head layers: 13
    [ 0] n=1 Conv                 args=[64, 3, 2]
    [ 1] n=1 Conv                 args=[128, 3, 2]
    [ 2] n=1 C3k2PrunedBn         args=[256, False]
    [ 3] n=1 Conv                 args=[256, 3, 2]
    [ 4] n=1 C3k2PrunedBn         args=[512, False]
    [ 5] n=1 Conv                 args=[512, 3, 2]
    [ 6] n=1 C3k2Pruned           args=[512, True]
    [ 7] n=1 Conv                 args=[1024, 3, 2]
    [ 8] n=1 C3k2Pruned           args=[1024, True]
    [ 9] n=1 SPPFPruned           args=[1024, 5, 3, True]
    [10] n=1 C2PSAPruned          args=[1024]
    [11] n=1 nn.Upsample          args=['None', 2, 'nearest']
    [12] n=1 Concat               args=[1]
    [13] n=1 C3k2Pruned           args=[512, True]
    [14] n=1 nn.Upsample          args=['None', 2, 'n

## 7. Finetune + CWD

In [7]:
if n_base < EPOCHS or not PRUNED.exists():
    print("Bo qua: chua co model da prune.")
    n_ours = 0
else:
    # Teacher la baseline cua chinh size nay.
    n_ours = train(OURS_NAME, str(PRUNED),
                   finetune=True,          # build DetectionModelPruned tu maskbndict
                   kd=True, kd_teacher=str(BEST_BASE), kd_method="cwd",
                   kd_lambda=0.5, kd_layers="neck", kd_warmup=5,
                   cwd_temperature=9.0)
    if n_ours < EPOCHS:
        print()
        print("Ours moi {}/{} epoch - Add Data output lan nay roi chay lai."
              .format(n_ours, EPOCHS))

[vd_ourss] train tu dau
New https://pypi.org/project/ultralytics/8.4.163 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
                                                      CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, cwd_learnable_tau_init=9.0, cwd_learnable_tau_lr=0.001, cwd_projection=False, cwd_tau_reg=0.1, cwd_temperature=9.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dms=False, dms_decay_ratio=1, dms_grad_scale=-1.0, dms_lambda=1.0, dms_lr=2e-05, dms_target=0.3, dms_taylor_type=taylor, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, finetune=True, 

## 8. Ket qua

In [8]:
def measure(name):
    best = REPO_DIR / "runs" / name / "weights" / "best.pt"
    if done_epochs(name) < EPOCHS or not best.exists():
        return None
    m = YOLO(str(best))
    r = m.val(data=DATA, imgsz=IMGSZ, batch=BATCH, device=DEVICE.split(",")[0])
    return (sum(p.numel() for p in m.model.parameters()) / 1e6,
            r.box.map50 * 100, r.box.map * 100)

rows = [("YOLO26-" + SIZE.upper(), measure(BASE_NAME)),
        ("Ours-" + SIZE.upper(), measure(OURS_NAME))]

print()
if all(v for _, v in rows):
    print("| Model | Params (M) | AP50 | AP50-95 |")
    print("|---|---:|---:|---:|")
    for label, (par, ap50, ap) in rows:
        print("| {} | {:.2f} | {:.2f} | {:.2f} |".format(label, par, ap50, ap))
    print()
    print("XONG ca hai. Gui lai bang tren.")
else:
    print("baseline:", done_epochs(BASE_NAME), "/", EPOCHS, "epoch")
    print("ours    :", done_epochs(OURS_NAME), "/", EPOCHS, "epoch")
    print()
    print("CHUA XONG - Add Data output lan nay roi Save & Run All lai.")

Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 122 layers, 9,469,050 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2446.0±489.9 MB/s, size: 131.3 KB)
val: Scanning /kaggle/working/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 153.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.6it/s 9.7s
                   all        548      38759      0.525      0.395      0.414      0.248
            pedestrian        520       8844       0.57       0.44      0.477       0.22
                people        482       5125      0.525      0.325      0.358      0.144
               bicycle        364       1287      0.328      0.155      0.153     0.0658
                   car        515      14064      0.718      0.791      0.808      0.572
                   van       